# Northern Shift Guard — Model Notebook
**Team:** NorthMind | **Event:** Cursor Hackathon Sudbury 2026

Steps:
1. Setup & imports
2. Test Replicate vision on sample images
3. Prompt tuning → stable JSON
4. Nemotron supervisor action
5. Export config to backend

In [1]:
# Cell 2 — Load env
import os
from dotenv import load_dotenv
load_dotenv('../.env')

REPLICATE_TOKEN = os.getenv('REPLICATE_API_TOKEN')
NVIDIA_KEY = os.getenv('NVIDIA_API_KEY')

print('Replicate token set:', bool(REPLICATE_TOKEN))
print('NVIDIA key set:', bool(NVIDIA_KEY))

Replicate token set: True
NVIDIA key set: False


In [2]:
# Cell 3 — Load a sample image as base64 data URL
import base64
from pathlib import Path

def image_to_data_url(path: str) -> str:
    with open(path, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    ext = Path(path).suffix.lstrip('.')
    mime = 'jpeg' if ext in ('jpg', 'jpeg') else ext
    return f'data:image/{mime};base64,{data}'

# List available sample images
samples = list(Path('../sample_images').glob('*'))
print('Sample images found:', [s.name for s in samples])

# Use first image or set manually
if samples:
    TEST_IMAGE = str(samples[0])
else:
    TEST_IMAGE = '../sample_images/test.jpg'  # add your own image here
print('Using:', TEST_IMAGE)

Sample images found: ['pass_compliant.jpg', 'README.md', 'fatigue_tired_operator.jpg', 'fail_missing_hardhat.jpg']
Using: ../sample_images/pass_compliant.jpg


In [3]:
# Cell 4 — System prompt
SYSTEM_PROMPT = """You are a mining site safety AI for Northern Ontario operations.

Analyze the uploaded image of a worker and return ONLY valid JSON in this exact schema:

{
  \"hard_hat\": \"pass\" | \"fail\" | \"unclear\",
  \"hi_vis\": \"pass\" | \"fail\" | \"unclear\",
  \"fatigue_risk\": \"low\" | \"medium\" | \"high\" | \"unclear\",
  \"evidence\": [\"<specific observation 1>\", \"<specific observation 2>\"],
  \"overall_status\": \"pass\" | \"fail\" | \"unclear\"
}

Rules:
- hard_hat: pass only if a hard hat is clearly visible on the worker's head
- hi_vis: pass only if a high-visibility vest or jacket is visible on the torso
- fatigue_risk: assess visible cues only (drooping eyelids, head position, posture)
- evidence: 2-4 specific, factual observations
- overall_status: fail if either hard_hat or hi_vis is fail; pass only if both pass
- Return ONLY the JSON. No markdown, no explanation."""

In [4]:
# Cell 5 — Run Replicate vision model
import replicate
import json

os.environ['REPLICATE_API_TOKEN'] = REPLICATE_TOKEN

def run_vision(image_path: str) -> dict:
    data_url = image_to_data_url(image_path)
    
    output = replicate.run(
        'yorickvp/llava-13b:b5f6212d032508382d61ff00469ddda3e32fd8a0755a17f6664032f16c9a0a07',
        input={
            'image': data_url,
            'prompt': SYSTEM_PROMPT,
            'max_tokens': 512,
            'temperature': 0.1,
        }
    )
    raw = ''.join(output)
    print('Raw output:\n', raw)
    
    # Parse JSON
    clean = raw.strip()
    if clean.startswith('```'):
        clean = clean.split('```')[1]
        if clean.startswith('json'):
            clean = clean[4:]
    return json.loads(clean.strip())

result = run_vision(TEST_IMAGE)
print('\nParsed result:')
print(json.dumps(result, indent=2))

ReplicateError: ReplicateError Details:
title: Invalid version or not permitted
status: 422
detail: The specified version does not exist (or perhaps you don't have permission to use it?)

In [ ]:
# Cell 6 — Nemotron supervisor action
from openai import OpenAI

SAFETY_REF = """Ontario Regulation 854 (Mines and Mining Plants):
- Section 81: Every worker in a mine shall wear protective headgear at all times in areas where there is risk of head injury.
- Section 79: High-visibility apparel required in active work zones and around mobile equipment.
- Fatigue: Workers showing visible signs of fatigue should be assessed before operating heavy machinery."""

def get_supervisor_action(scan_result: dict) -> str:
    client = OpenAI(
        base_url='https://integrate.api.nvidia.com/v1',
        api_key=NVIDIA_KEY,
    )
    prompt = f"""You are a shift supervisor AI for a Northern Ontario mining site.
Given this PPE and fatigue scan, write ONE prioritized action for the supervisor.
Be direct, safety-first. Max 2 sentences.

Scan result:
{json.dumps(scan_result, indent=2)}

Safety reference:
{SAFETY_REF}

Supervisor action:"""
    
    response = client.chat.completions.create(
        model='nvidia/llama-3.1-nemotron-70b-instruct',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=200,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

action = get_supervisor_action(result)
print('Supervisor action:')
print(action)

In [ ]:
# Cell 7 — Visualize result
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

STATUS_COLORS = {'pass': 'green', 'fail': 'red', 'unclear': 'orange', 'low': 'green', 'medium': 'orange', 'high': 'red'}

img = Image.open(TEST_IMAGE)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Image
axes[0].imshow(img)
axes[0].axis('off')
axes[0].set_title('Input Image', fontsize=13)

# Results panel
axes[1].axis('off')
y = 0.95
axes[1].text(0.05, y, 'Northern Shift Guard — Scan Result', fontsize=13, fontweight='bold', transform=axes[1].transAxes)
y -= 0.1

for label, key in [('Hard Hat', 'hard_hat'), ('Hi-Vis Vest', 'hi_vis'), ('Fatigue Risk', 'fatigue_risk'), ('Overall', 'overall_status')]:
    val = result.get(key, 'unclear')
    color = STATUS_COLORS.get(val, 'gray')
    axes[1].text(0.05, y, f'{label}:', fontsize=11, transform=axes[1].transAxes)
    axes[1].text(0.4, y, val.upper(), fontsize=11, color=color, fontweight='bold', transform=axes[1].transAxes)
    y -= 0.08

y -= 0.05
axes[1].text(0.05, y, 'Evidence:', fontsize=11, fontweight='bold', transform=axes[1].transAxes)
y -= 0.07
for ev in result.get('evidence', []):
    axes[1].text(0.05, y, f'• {ev}', fontsize=9, wrap=True, transform=axes[1].transAxes)
    y -= 0.07

y -= 0.05
axes[1].text(0.05, y, 'Supervisor Action:', fontsize=11, fontweight='bold', transform=axes[1].transAxes)
y -= 0.07
axes[1].text(0.05, y, action, fontsize=9, wrap=True, transform=axes[1].transAxes, color='navy')

plt.tight_layout()
plt.savefig('../sample_images/last_scan_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to sample_images/last_scan_result.png')

In [ ]:
# Cell 8 — Export config to backend (run after you're happy with results)
import json
from pathlib import Path

config = {
    'replicate_model': 'yorickvp/llava-13b:b5f6212d032508382d61ff00469ddda3e32fd8a0755a17f6664032f16c9a0a07',
    'nemotron_model': 'nvidia/llama-3.1-nemotron-70b-instruct',
    'system_prompt': SYSTEM_PROMPT,
    'confidence_thresholds': {'fatigue_high': 0.8, 'fatigue_medium': 0.5},
    'notes': 'Tuned in notebook — llava-13b with structured JSON prompt'
}

config_path = Path('../backend/config/model_config.json')
config_path.write_text(json.dumps(config, indent=2))
print(f'Config exported to {config_path}')